Making the training pipeline 

with dataloading and overfitting it just to verify that the model actually learns !

In [ ]:
import torch 
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader 

class Custom_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,kernel_size = 3 , stride = 1, padding = 0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),

            nn.Conv2d(32,64,kernel_size=3 , stride = 1, padding = 0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        self.classifier =nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*5*5,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
dataset= datasets.MNIST(root = './data', train = True, transform = transforms.ToTensor())
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

model=Custom_CNN()
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)

for epoch in range(5):
    total_loss = 0
    for image,labels in dataloader:
        output=model(image)
        Loss=loss_fn(output,labels)
        total_loss += Loss.item()

        optimizer.zero_grad()
        Loss.backward()
        optimizer.step()
        print(f'Epoch: {epoch+1}, Loss: {Loss.item():.4f}')
    print(f'Epoch: {epoch+1}, Total Loss: {total_loss}')



Testing the predictio for the same data set

In [7]:
image,labels = dataset[0]
print(f'Image shape: {image.shape}, Label: {labels}')
output = model(image.unsqueeze(0))
predicted_label = torch.argmax(output, dim=1).item()    
print(f'Predicted Label: {predicted_label}')    


Image shape: torch.Size([1, 28, 28]), Label: 5
Predicted Label: 5


In [8]:
correct = 0
total = 0

for images, labels in dataloader:
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

    correct += (predicted == labels).sum().item()
    total += labels.size(0)

print("Accuracy:", correct / total)

Accuracy: 0.9935333333333334
